# 25 — Context-Aware SQR Calibration

## Purpose

Calibrate Selective Quantum-gate by Rotation (SQR) gates — the native entangling operations between the qubit and the bosonic storage mode. The full pipeline runs from unitary decomposition through closed-loop calibration, process tomography, fidelity benchmarking, context-dependent corrections, and final validation.

## Methodology

1. **SQR gate decomposition** — decompose a target unitary into an optimal SQR gate sequence using the calibration orchestrator
2. **Closed-loop calibration** — iteratively tune SQR gate parameters on hardware using feedback from measurement outcomes
3. **Process tomography** — reconstruct the full process matrix (χ matrix) of the calibrated SQR gate
4. **Fidelity benchmarking** — run gate set tomography or randomized benchmarking with SQR gates to extract average gate fidelity
5. **Context-aware update** — apply corrections that account for cross-talk, frequency crowding, and Fock-state-dependent shifts
6. **Final validation** — end-to-end validation of the complete calibrated SQR gate set

## Expected Outcomes

- SQR gate fidelity > 99% after closed-loop calibration
- Process matrix consistent with the target unitary (process fidelity > 98%)
- Context-aware corrections improving multi-gate sequence fidelity by 0.5–2%

## Prerequisites

- **Notebook 21** — storage cavity characterized (frequency, χ per Fock state)
- **Notebook 23** — SNAP gates calibrated (SQR decomposition uses SNAP)
- **Notebook 14** — single-qubit gate benchmarking (validates underlying qubit operations)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the free-evolution tomography checkpoint from notebook 24.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    CalibrationOrchestrator,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="25_context_aware_sqr_calibration",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="24_free_evolution_tomography",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 24 status: {prev_checkpoint['status']}")

## 2. Configuration — SQR Calibration Defaults

Set the maximum decomposition rank, gate duration, and averaging count used across all calibration steps.

In [ ]:
SQR_N_AVG = 10000
SQR_MAX_RANK = 3
SQR_DURATION_NS = 1000

print("SQR calibration settings:")
print(f"  n_avg: {SQR_N_AVG}")
print(f"  max_rank: {SQR_MAX_RANK}")
print(f"  duration: {SQR_DURATION_NS} ns")

## 3. Execution — SQR Gate Decomposition

Decompose the target unitary into an SQR gate sequence. The decomposition rank controls the approximation quality vs. circuit depth trade-off.

In [ ]:
orchestrator = CalibrationOrchestrator(session)

decomposition_result = orchestrator.decompose_unitary(
    max_rank=SQR_MAX_RANK,
    duration_ns=SQR_DURATION_NS,
)

print("SQR decomposition complete.")
print(f"  Number of SQR gates: {decomposition_result.get('n_gates', 'N/A')}")

## 4. Execution — SQR Closed-Loop Calibration

Iteratively tune the SQR gate parameters on hardware using measurement feedback. Each iteration adjusts pulse amplitudes and phases to minimize the gate error.

In [ ]:
sqr_cal_result = orchestrator.run_closed_loop_calibration(
    n_avg=SQR_N_AVG,
)

print("SQR closed-loop calibration complete.")

## 5. Verification — SQR Process Tomography

Reconstruct the full process matrix (χ matrix) of the calibrated SQR gate. Compare to the ideal unitary to extract the process fidelity.

In [ ]:
sqr_tomo_result = orchestrator.run_process_tomography(
    n_avg=SQR_N_AVG,
)

print("SQR process tomography complete.")

## 6. Verification — SQR Fidelity Benchmarking

Run gate set tomography or randomized benchmarking using the calibrated SQR gates to extract the average gate fidelity and identify dominant error channels.

In [ ]:
sqr_fidelity_result = orchestrator.run_fidelity_benchmarking(
    n_avg=SQR_N_AVG,
)

print("SQR fidelity benchmarking complete.")

## 7. Execution — SQR Context-Aware Parameter Update

Apply context-dependent corrections that account for cross-talk, AC Stark shifts, and Fock-state-dependent frequency errors in multi-gate sequences.

In [ ]:
sqr_context_result = orchestrator.apply_context_corrections(
    n_avg=SQR_N_AVG,
)

print("SQR context-aware update complete.")

## 8. Verification — SQR Final Validation

End-to-end validation of the complete calibrated SQR gate set, including context-aware corrections. This is the gate-keeper for all downstream Hamiltonian simulation experiments.

In [ ]:
sqr_validation_result = orchestrator.run_final_validation(
    n_avg=SQR_N_AVG,
)

print("SQR final validation complete.")

## 9. Checkpoint — Save SQR Calibration Results

Persist the full SQR calibration (decomposition, closed-loop parameters, context corrections). Downstream Trotterization and cluster-state experiments depend on these calibrated gates.

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="25_context_aware_sqr_calibration",
    status="calibrated",
    summary="Full SQR calibration pipeline: decomposition, closed-loop cal, process tomo, benchmarking, context corrections, validation.",
    consumed_inputs={
        "sqr_n_avg": SQR_N_AVG,
        "sqr_max_rank": SQR_MAX_RANK,
        "sqr_duration_ns": SQR_DURATION_NS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="26_sequential_simulation",
    notes=[
        "This notebook drives the CalibrationOrchestrator end-to-end.",
        "Context-aware corrections are essential for multi-gate sequence fidelity.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")